# Syndrome coding in the spatial domain

## Module imports

In [23]:
import textwrap
import numpy as np
from PIL import Image
from Crypto.Hash import SHA256
from math import log10, sqrt

## Functions for secret message file processing

In [24]:
# convert byte array to bit string
def bytes_to_bits(bytes):
            
    bits = ''.join(f"{byte:08b}" for byte in bytes)

    return bits


# convert bit string to byte array
def bits_to_bytes(bits):

    val = int(bits, 2)
    n_B = len(bits) // 8

    return val.to_bytes(n_B, byteorder="big")


# zero pad and wrap to equal k length blocks
def pad_wrap(bits, k):

    mod = len(bits) % k
    if mod != 0:
        bits += "0" * (k-mod)
    
    return textwrap.wrap(bits, k)


# merge binary blocks to string and unpad 0s
def merge_unpad(bits):

    bits = "".join(bits)
    mod = len(bits) % 8

    return bits[:len(bits) - mod]

## Image and syndrome processing functions

In [25]:
def PSNR(original, noisy):

    mse = np.mean((original-noisy) ** 2)

    # set rounded value for PSNR
    # MSE = 0 --> PSNR goes to infinity
    if(mse == 0):

        return 100, 0
    
    max_pixel = 255.0
    psnr = 20 * log10(max_pixel / sqrt(mse))
    
    return psnr, mse

        
# get syndrome per binary block of bits
def get_syndrome(bits):
    
    syn = 0
    for i, bit in enumerate(bits):
        if bit:
            syn ^= i
            
    return syn


# 1st method
# LSB substitution method
def substitute_lsb(pv, b):

    pv &= ~ np.uint8(1)
    pv |=   np.uint8(b)

    return pv


# 2nd method
# LSB +/-1 embedding
def match_lsb(pv):

    match pv:
        case 0:
            pv += np.uint8(1)
        case 255:
            pv -= np.uint8(1)
        case _:
            pv += np.random.choice([-1, 1])

    return pv

## Main ```embed_pixels``` and ```extract_pixels``` functions

In [26]:
def embed_pixels(array, ids, bits):

    # iteration per binary and indices blocks
    for b, i in zip(bits, ids):

        # block of pixel values
        pvs = array[i]
        # LSB cover block
        c = pvs % 2
        
        syn = get_syndrome(c) 
        # XOR with binary block for embedding
        err = syn ^ int(b, 2)

        # err==0: syn equals the binary block
        # err!=0: modify pixel for syn coding
        if err != 0:

            # i - indices block in iteration
            # err - index in an array of pixel indices i 
            # mod - pixel index for LSB substitution
            mod = i[err]
            pv = array[mod]
            pv1 = match_lsb(pv)
            array[mod] = pv1

    return array


def extract_pixels(array, ids, k):
    
    bits = []

    for i in ids:

        # block of pixel values
        pvs = array[i]
        # LSB stego block
        s = pvs % 2
        # syn is embedded binary block
        syn = get_syndrome(s)
        # zfill adds to block k length
        b = bin(syn)[2:].zfill(k)
        bits.append(b)
        
    return bits

## Embedding process

### Part 1: secret message file processing

In [27]:
f_name = b"message.txt"

with open(f_name, "rb") as f:
    data = f.read()

print("content: ", data)

# part 2 message bytes: name + contents
p2_B = len(f_name).to_bytes(1) + f_name + data
# part 1 message bytes: p2_B size stored in 3B
p1_B = len(p2_B).to_bytes(3)

print("p1_B: ", p1_B)
print("p2_B: ", p2_B)

# set data bin block length for syndrome coding
k = 4
# reserved block length
b_len = 2 ** k

# part 1 bits
p1_b = bytes_to_bits(p1_B)
p1_b = pad_wrap(p1_b,k)

# part 2 bits
p2_b = bytes_to_bits(p2_B)
p2_b = pad_wrap(p2_b,k)

# n of blocks per parts 1 and 2
n_b1 = len(p1_b)
n_b2 = len(p2_b)

# n of points per parts 1 and 2
n_pts1 = n_b1 * b_len
n_pts2 = n_b2 * b_len

# indices shape for iteration over blocks
shape_ids1 = (n_b1, b_len)
shape_ids2 = (n_b2, b_len)

content:  b'"Listen very carefully, I shall say this only once!"\n\n                                   - Michele Dubois, \'Allo \'Allo!\n'
p1_B:  b'\x00\x00\x84'
p2_B:  b'\x0bmessage.txt"Listen very carefully, I shall say this only once!"\n\n                                   - Michele Dubois, \'Allo \'Allo!\n'


### Part 2: image processing

In [28]:
cover = Image.open("lenna.bmp")
c_arr = np.array(cover)
c_arr1 = c_arr.flatten()

rng_passw = b"stegopass"
h = SHA256.new(data= rng_passw)

seed = int.from_bytes(h.digest())
rng = np.random.default_rng(seed)

# n of image pixel points
n_pts = len(c_arr1)
# initial indices array
ids_arr0 = np.arange(n_pts)

# indices selection for embedding p1_b
sel1 = rng.choice(ids_arr0, n_pts1, 0)

# indices selection for embedding p2_b
mask = ~np.isin(ids_arr0, sel1)
ids_arr1 = ids_arr0[mask]
sel2 = rng.choice(ids_arr1, n_pts2, 0)

sel1 = np.reshape(sel1, shape_ids1)
sel2 = np.reshape(sel2, shape_ids2)

# flat stego pixels array after embedding
s_arr1 = embed_pixels(c_arr1, sel1, p1_b)
s_arr1 = embed_pixels(c_arr1, sel2, p2_b)

shape = c_arr.shape
s_arr = s_arr1.reshape(shape)

stego = Image.fromarray(s_arr)
stego.save("stego3.png")

## Extraction

### Part 1: extracting `p1_B` from stego image

In [29]:
stego = Image.open("stego3.png")
s_arr = np.array(stego)
s_arr1 = s_arr.flatten()

rng_passw = b"stegopass"
h = SHA256.new(data= rng_passw)

seed = int.from_bytes(h.digest())
rng = np.random.default_rng(seed)

# n of image pixel points
n_pts = len(s_arr1)
# initial indices array
ids_arr0 = np.arange(n_pts)

# set data bin block length for syndrome coding
k = 4
# reserved block length
b_len = 2 ** k

# part 1: p2_B length extraction (3 B)
#########################################
# n of embedded blocks for p1_b
n_b1 = (3*8//k + int(3*8%k !=0))
# n of embedded points for p1_b
n_pts1 = b_len * n_b1

shape_ids1 = (n_b1, b_len)

# indices selection with embedded p1_b
sel1 = rng.choice(ids_arr0, n_pts1, 0)
sel1 = np.reshape(sel1, shape_ids1)

p1_b = extract_pixels(s_arr1, sel1, k)
p1_B = bits_to_bytes(merge_unpad(p1_b))

size = int.from_bytes(p1_B)
print(size)

132


### Part 2: extracting `p2_B` from stego image

It is important to run the following cell only once after the previous part! Otherwise, the pseudo random number generator will generate different positions each time (resulting in extraction of random bytes for p2_B). If this happens, run the Part 1 again to initialize the RNG and start over (or simply merge part 1 and part 2 in a single cell)

In [30]:
# n of embedded blocks for p2_b
n_b2 = (size*8//k + int(size*8%k !=0))
# n of embedded points for p2_b
n_pts2 = b_len * n_b2

shape_ids2 = (n_b2, b_len)

# indices selection with embedded p2_b
mask = ~np.isin(ids_arr0, sel1)
ids_arr1 = ids_arr0[mask]
sel2 = rng.choice(ids_arr1, n_pts2, 0)

sel2 = np.reshape(sel2, shape_ids2)

p2_b = extract_pixels(s_arr1, sel2, k)
data = bits_to_bytes(merge_unpad(p2_b))

In [31]:
name_l = data[0]
f_name = data[1:1+name_l]
f_data = data[1+name_l: ]
print(f"extracted file names is: {f_name}")
print(f"extracted file content is: \n{f_data}")

extracted file names is: b'message.txt'
extracted file content is: 
b'"Listen very carefully, I shall say this only once!"\n\n                                   - Michele Dubois, \'Allo \'Allo!\n'


## Signal analysis

In [32]:
cover = Image.open("lenna.bmp")
c_arr = np.asarray(cover)

stego = Image.open("stego3.png")
s_arr = np.asarray(stego)

psnr, mse = PSNR(c_arr, s_arr)

print(f"PSNR value: {psnr}")
print(f"MSE value: {mse}")

PSNR value: 83.02201357105278
MSE value: 0.000324249267578125


## Cover image embedding capacity

In [33]:
cover = Image.open("lenna.bmp")
c_arr = np.asarray(cover)
c_arr1 = c_arr.flatten()

n_pts = len(c_arr1)

# set data bin block length for syndrome coding
k = 4
# reserved block length
b_len = 2 ** k

# cap in n of blocks
n_b = n_pts // b_len

# cap in n of bytes
byte_cap = k * n_b // 8

print(f"cover capacity: {byte_cap} B")

cover capacity: 24576 B
